In [1]:
import pandas as pd
import optuna
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import os
from dotenv import load_dotenv
from supabase import create_client, Client

c:\Users\ardel\.conda\envs\ml_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
URL = os.getenv("SUPABASE_URL")
KEY = os.getenv("SUPABASE_KEY")
BUCKET_NAME = os.getenv("SUPABASE_BUCKET")
supabase: Client = create_client(URL, KEY)

In [3]:
cloud_path = f"{URL}/storage/v1/object/public/{BUCKET_NAME}/ulasan_ikd_preprocessed.csv"
print(f"Loading data from Supabase: {cloud_path}")

df = pd.read_csv(cloud_path)

# [KOREKSI FATAL]: Kembalikan NaN menjadi string kosong khusus untuk kolom ablasi
df = df.fillna("")

print("Data loaded successfully. Shape:", df.shape)

X = df['text_baseline']
y = df['label']

Loading data from Supabase: https://ffygtotdlnxhscukpmpu.supabase.co/storage/v1/object/public/dataset-ikd/ulasan_ikd_preprocessed.csv
Data loaded successfully. Shape: (34902, 7)


In [4]:
# 2. Data Splitting (Train: 64%, Val: 16%, Test: 20%)
# Train & Val digunakan untuk Optuna. Test disimpan murni untuk evaluasi akhir.
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp)

print("2. Vektorisasi TF-IDF...")
tfidf = TfidfVectorizer()
X_train_vec = tfidf.fit_transform(X_train)
X_val_vec = tfidf.transform(X_val)
X_test_vec = tfidf.transform(X_test)

2. Vektorisasi TF-IDF...


In [6]:
# 3. Fungsi Objektif Optuna
def objective(trial):
    # ==========================================
    # 1. Ruang Pencarian Hyperparameter TF-IDF
    # ==========================================
    ngram_name = trial.suggest_categorical('ngram_range', ['unigram', 'bigram'])
    ngram_param = (1, 1) if ngram_name == 'unigram' else (1, 2)
    
    min_df_param = trial.suggest_int('min_df', 1, 5)
    max_df_param = trial.suggest_float('max_df', 0.7, 1.0)
    
    tfidf = TfidfVectorizer(
        ngram_range=ngram_param,
        min_df=min_df_param,
        max_df=max_df_param
    )
    
    X_train_vec = tfidf.fit_transform(X_train)
    X_val_vec = tfidf.transform(X_val)
    
    # ==========================================
    # 2. Ruang Pencarian Hyperparameter SVM
    # ==========================================
    c_param = trial.suggest_float('C', 0.1, 10.0, log=True)
    kernel_param = trial.suggest_categorical('kernel', ['linear', 'rbf'])
    
    if kernel_param == 'rbf':
        gamma_param = trial.suggest_categorical('gamma', ['scale', 'auto'])
        model = SVC(C=c_param, kernel=kernel_param, gamma=gamma_param, random_state=42)
    else:
        model = SVC(C=c_param, kernel=kernel_param, random_state=42)
        
    # ==========================================
    # 3. Training & Evaluasi
    # ==========================================
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_val_vec)
    macro_f1 = f1_score(y_val, y_pred, average='macro')
    
    return macro_f1

print("3. Memulai Hyperparameter Tuning dengan Optuna...")
db_file = "svm_tuning.sqlite3"
storage_url = f"sqlite:///{db_file}"
study_name = "SVM_IKD_Tuning_v3"

study = optuna.create_study(
    direction='maximize', 
    study_name=study_name, 
    storage=storage_url, 
    load_if_exists=True  
)

# Jalankan 20 trial
study.optimize(objective, n_trials=20)

print("\n=== HASIL TERBAIK OPTUNA ===")
print(f"Best Macro F1 (Validation): {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-10-28 22:10:25,031] A new study created in RDB with name: SVM_IKD_Tuning_v3


3. Memulai Hyperparameter Tuning dengan Optuna...


[I 2026-10-28 22:10:47,207] Trial 0 finished with value: 0.9220469498480155 and parameters: {'ngram_range': 'bigram', 'min_df': 3, 'max_df': 0.7515753508451047, 'C': 1.0564497159136366, 'kernel': 'linear'}. Best is trial 0 with value: 0.9220469498480155.
[I 2026-10-28 22:11:15,897] Trial 1 finished with value: 0.37284913071616516 and parameters: {'ngram_range': 'unigram', 'min_df': 3, 'max_df': 0.7994352105070033, 'C': 1.392390843274786, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 0 with value: 0.9220469498480155.
[I 2026-10-28 22:11:34,940] Trial 2 finished with value: 0.9141635987674122 and parameters: {'ngram_range': 'bigram', 'min_df': 2, 'max_df': 0.9947502666496744, 'C': 0.2116579099901444, 'kernel': 'linear'}. Best is trial 0 with value: 0.9220469498480155.
[I 2026-10-28 22:11:57,880] Trial 3 finished with value: 0.9209139238017418 and parameters: {'ngram_range': 'bigram', 'min_df': 2, 'max_df': 0.8702575214401334, 'C': 0.5144343976485034, 'kernel': 'linear'}. Best is trial


=== HASIL TERBAIK OPTUNA ===
Best Macro F1 (Validation): 0.9248
Best Parameters:
  ngram_range: bigram
  min_df: 3
  max_df: 0.9737477221415765
  C: 1.0826203794707816
  kernel: rbf
  gamma: scale


In [8]:
# 4. Evaluasi Final Menggunakan Parameter Terbaik di Data Test
print("\n4. Melatih ulang model terbaik pada seluruh data training & evaluasi di Data Test (Data Buta)...")
best_params = study.best_params

if best_params['kernel'] == 'rbf':
    best_svm = SVC(C=best_params['C'], kernel=best_params['kernel'], gamma=best_params['gamma'], random_state=42)
else:
    best_svm = SVC(C=best_params['C'], kernel=best_params['kernel'], random_state=42)

# Fit menggunakan gabungan Train + Val agar model mendapat data maksimal sebelum Test
X_train_val_vec = tfidf.fit_transform(X_temp)
X_test_final_vec = tfidf.transform(X_test)

best_svm.fit(X_train_val_vec, y_temp)
y_pred_final = best_svm.predict(X_test_final_vec)
final_macro_f1 = f1_score(y_test, y_pred_final, average='macro')

print("\n=== CLASSIFICATION REPORT (DATA TEST) ===")
print(classification_report(y_test, y_pred_final, digits=4))
print(f"Skor Uji Ablasi (Default) : 0.9269")
print(f"Skor Final Optuna (Tuned) : {final_macro_f1:.4f}")


4. Melatih ulang model terbaik pada seluruh data training & evaluasi di Data Test (Data Buta)...

=== CLASSIFICATION REPORT (DATA TEST) ===
              precision    recall  f1-score   support

           0     0.9111    0.9743    0.9417      4125
           1     0.9588    0.8627    0.9082      2856

    accuracy                         0.9287      6981
   macro avg     0.9349    0.9185    0.9249      6981
weighted avg     0.9306    0.9287    0.9280      6981

Skor Uji Ablasi (Default) : 0.9269
Skor Final Optuna (Tuned) : 0.9249


In [9]:
print(f"Mengunggah riwayat eksperimen ({db_file}) ke Supabase...")
with open(db_file, 'rb') as f:
    supabase.storage.from_(BUCKET_NAME).upload(
        file=f, 
        path=db_file, 
        file_options={"x-upsert": "true"}
    )
print("✅ Riwayat Tuning SVM berhasil diamankan ke Supabase. Tim dapat mengaksesnya sekarang.")

Mengunggah riwayat eksperimen (svm_tuning.sqlite3) ke Supabase...
✅ Riwayat Tuning SVM berhasil diamankan ke Supabase. Tim dapat mengaksesnya sekarang.
